# Player Tracking in Sports - 3D Geometry Evaluation

This notebook evaluates the **3D geometry quality** of the multi-view tracking pipeline.

The evaluation works by projecting the annotated GT `TriangulationOutput` back into each camera's 2D pixel space via `cv2.projectPoints`, then comparing the projected positions against the per-camera predicted `TrackingOutput` produced by the YOLO + DeepSORT pipeline.

Since identities are already aligned by the algorithm (`class_name` ↔ `track_id`), no spatial matching is needed — each GT point is looked up directly by identity.

Two metric families are computed per camera:
- **ReprojectionMetrics** — mean/median/RMSE reprojection error and accuracy at 2/5/10 px thresholds.
- **TrajectoryMetrics** — ADE, FDE, MTE, smoothness and jitter over full per-identity trajectories.

---

### Imports

In [ ]:
from pathlib import Path

from src.utils.annotations.load_annotations import load_annotations
from src.utils.eval_tables import show_reprojection_table, show_trajectory_table
from src.calibration.camera_data import CameraData
from src.types.tracking import TrackingOutput
from src.geometry.rectification import rectify_tracking_output
from src.geometry.triangulation import triangulate_rectified_outputs

from src.evaluation.evaluate_geometry import evaluate_geometry

### Global Parameters

Global parameter variables configure experiment scope and file locations, enabling quick tests, dataset switching, and evaluation-scope changes without modifying implementation.

In [ ]:
INSPECT_CAMERA_ID = "cam_13"                                # The camera we will inspect in detail
FRAME_STRIDE      = 5                                       # Step between GT annotation frames and prediction frame indices

CAMERA_SETTINGS_DIR = "data/camera_settings"                # Directory where the camera settings JSON files are stored
ANNOTATIONS         = "data/annotations/tracking_01"        # Directory where the tracking annotations are stored
TRACKING_RESULTS_DIR = "results/tracking"                   # Root holding {camera}/serialized/{camera}/tracking.json

# Cameras used for triangulation and geometry evaluation
CAMERAS = {
    "cam_13": {"camera_settings": f"{CAMERA_SETTINGS_DIR}/cam13_settings.json"},
    #"cam_2":  {"camera_settings": f"{CAMERA_SETTINGS_DIR}/cam2_settings.json"},
    "cam_4":  {"camera_settings": f"{CAMERA_SETTINGS_DIR}/cam4_settings.json"},
}

---

# Load Annotations

Load the ground-truth tracking annotations. These are the reference labels against which the pipeline output is evaluated.
The annotations contain per-camera `TrackingOutput` objects with `class_name` as the GT identity key (e.g. `"White_14"`).

In [ ]:
ANNOTATIONS_VERSION = Path(ANNOTATIONS).name   # e.g. 'tracking_01'

ground_truth = {}
for cam_id in CAMERAS:
    ground_truth[cam_id] = load_annotations(cam_id, ANNOTATIONS_VERSION)
    n_frames = len(ground_truth[cam_id].frames)
    n_dets   = sum(len(f.detections) for f in ground_truth[cam_id].frames)
    print(f"[{cam_id}] {n_frames} annotated frames, {n_dets} total detections")

---

# Load Predicted Tracking Output

Load the per-camera tracking results produced by the YOLO + DeepSORT pipeline (`run_2D_pipeline.py`).
Each camera's JSON is read from `results/tracking/{camera_id}/serialized/{camera_id}/tracking.json`.

In [ ]:
tracking_result: dict[str, TrackingOutput] = {}

for cam_id in CAMERAS:
    path = Path(TRACKING_RESULTS_DIR) / cam_id / "serialized" / cam_id / "tracking.json"
    if not path.exists():
        print(f"[{cam_id}] Tracking output not found at {path} — run run_2D_pipeline.py first.")
        continue
    tracking_result[cam_id] = TrackingOutput.read(str(path))
    n_frames = len(tracking_result[cam_id].frames)
    n_dets   = sum(len(f.detections) for f in tracking_result[cam_id].frames)
    print(f"[{cam_id}] Loaded {n_frames} frames, {n_dets} total detections")

---

# Build GT Triangulation

Triangulate the GT annotations into a `TriangulationOutput` — a sequence of 3D world-frame positions derived from the multi-view ground-truth tracking.
This is the reference 3D structure that will be projected back into each camera's image plane for evaluation.

In [ ]:
cameras = {cam_id: CameraData.load(cam_id) for cam_id in CAMERAS}

# Rectify GT tracking output for each camera (undistort bbox centres into normalised rays)
rectified = {
    cam_id: rectify_tracking_output(ground_truth[cam_id], cameras[cam_id])
    for cam_id in CAMERAS
}

# Triangulate across all cameras to produce GT 3D world positions
triangulation_result = triangulate_rectified_outputs(cameras, rectified)

print(f"GT triangulation: {len(triangulation_result.frames)} frames, "
      f"cameras: {triangulation_result.camera_ids}")

---

# 3D Geometry Evaluation

Evaluate 3D tracking quality by projecting the GT `TriangulationOutput` into each camera's 2D pixel space and comparing against the per-camera predicted `TrackingOutput`.

For each camera, the following metrics are computed:

## Reprojection Metrics

A GT identity's 3D world position is projected into pixel space via:

$$\mathbf{p} = K [R | t] \mathbf{P}_w$$

where $K$ is the intrinsic matrix, $[R | t]$ the extrinsics, and $\mathbf{P}_w$ the GT 3D world point. The reprojection error for a matched pair is then:

$$e = \| \hat{\mathbf{p}} - \mathbf{p}_{\text{pred}} \|_2$$

where $\mathbf{p}_{\text{pred}}$ is the predicted bbox centre. This gives a direct pixel-space measure of how accurately the 3D reconstruction back-projects onto each view.

- **Mean / Median / Std / RMSE** — distribution of reprojection errors across all matched pairs.
- **Accuracy @ N px** — fraction of matches with reprojection error below 2 / 5 / 10 pixels.
- **Matched / Unmatched** — coverage counts.

## Trajectory Metrics

Per-identity GT and predicted pixel trajectories are built and compared frame-by-frame:

- **ADE** (Average Displacement Error) — mean Euclidean distance between GT and predicted positions over all shared frames.
- **FDE** (Final Displacement Error) — Euclidean distance at the last shared frame; captures long-term drift.
- **MTE** (Median Trajectory Error) — median per-frame displacement, robust to outlier frames.
- **Smoothness** — mean frame-to-frame variation in predicted displacement magnitude; high values indicate erratic motion.
- **Jitter** — standard deviation of frame-to-frame displacements; captures high-frequency instability.
- **Fragments** — number of coverage gaps in the predicted track relative to the GT trajectory.

In [ ]:
geometry_results = evaluate_geometry(
    triangulations     = triangulation_result,
    annotated_tracking = rectified,
    frame_stride       = FRAME_STRIDE,
)

for cam_id, metrics in geometry_results.items():
    r = metrics.reprojection

---

## Reprojection Metrics

In [ ]:
show_reprojection_table(geometry_results)

---

## Trajectory Metrics

In [ ]:
show_trajectory_table(geometry_results)